# PART 1 — Getting Started with OpenAI


## Task 1: OpenAI Setup & Basic Prompt
1. Set up OpenAI API key.
2. Call an OpenAI chat model.
3. Send a simple prompt and print the response.


In [1]:
%pip install -qU langchain-openai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

API key loaded: True


In [3]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o-mini",temperature=0)

print("OpenAI chat model initialized successfully.")

OpenAI chat model initialized successfully.


In [4]:
response = chat_model.invoke("Explain what Generative AI is in simple terms, in 3 sentences.")

print(response.content)

Generative AI is a type of artificial intelligence that can create new content, such as text, images, music, or videos, based on patterns it has learned from existing data. It works by analyzing large amounts of information and then using that knowledge to generate original outputs that resemble the input data. Essentially, it's like a smart computer program that can invent new things by mimicking what it has seen before.


### observation 

Successfully configured the OpenAI API key and initialized the GPT-4o-mini chat model using LangChain's ChatOpenAI integration. Sent a basic prompt and received a generated response, confirming that the API connection and model invocation are working correctly.


# PART 2 — Retriever-Based RAG (Wikipedia + Vector Store)



## Task 2: Wikipedia Retriever

1. Use LangChain’s WikipediaRetriever.
2. Retrieve documents for a user query.
3. Print retrieved content.


In [5]:
%pip install -qU langchain-community wikipedia


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.retrievers import WikipediaRetriever

wiki_retriever = WikipediaRetriever(top_k_results=3,lang="en")

print("Wikipedia Retriever initialized successfully.")

Wikipedia Retriever initialized successfully.


/var/folders/30/m7yymk9x4t78ry_hq5dpf9dc0000gn/T/ipykernel_61126/291671863.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [8]:
query = "Lionel Messi"

wiki_docs = wiki_retriever.invoke(query)

print("Number of documents retrieved:", len(wiki_docs))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [9]:
import requests

url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "query",
    "list": "search",
    "srsearch": "Lionel Messi",
    "format": "json"
}

headers = {
    "User-Agent": "GenAI-Assignment/1.0 (educational project)"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=20
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response:", response.text[:500])

Status code: 200
Content type: application/json; charset=utf-8
Response: {"batchcomplete":"","continue":{"sroffset":10,"continue":"-||"},"query":{"searchinfo":{"totalhits":2151},"search":[{"ns":0,"title":"Lionel Messi","pageid":2150841,"size":330231,"wordcount":25110,"snippet":"<span class=\"searchmatch\">Lionel</span> Andr\u00e9s &quot;Leo&quot; <span class=\"searchmatch\">Messi</span> (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer","timestamp":"2026-11-16T19:06:53Z"},{"ns":0,"title":"Messi


In [10]:
query = "Lionel Messi"

wiki_docs = wiki_retriever.invoke(query)

print("Number of documents retrieved:", len(wiki_docs))

for i, doc in enumerate(wiki_docs, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:500])

Number of documents retrieved: 3

Result 1
Source: https://en.wikipedia.org/wiki/Lionel_Messi
Content: Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. Widely regarded as one of the greatest players in history, he played for the Argentina national team for 21 years. During his career, he has set numerous records for individual accolades won, including eight Ballons d'Or, six European Golden Shoes, and being named the world's best player by FIFA eight times. Messi was named the La

Result 2
Source: https://en.wikipedia.org/wiki/Messi%E2%80%93Ronaldo_rivalry
Content: The Messi–Ronaldo rivalry, or Ronaldo–Messi rivalry, is a sporting rivalry in football propelled by the media and fans that involves Argentine footballer Lionel Messi and Portuguese footballer Cristiano Ronaldo, mainly for being contemporaries and due to their similar records and sporting successes. They spe

In [11]:
for i, doc in enumerate(wiki_docs, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:1000])
    print("-" * 80)


Result 1
Source: https://en.wikipedia.org/wiki/Lionel_Messi
Content: Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. Widely regarded as one of the greatest players in history, he played for the Argentina national team for 21 years. During his career, he has set numerous records for individual accolades won, including eight Ballons d'Or, six European Golden Shoes, and being named the world's best player by FIFA eight times. Messi was named the Laureus World Sportsman of the Year in 2020 and 2023, and was named the All Time Men's World Best Player by the IFFHS in 2025. He was among Time's 100 most influential people in the world in 2011, 2012, and 2023.
Messi is the most decorated player in the history of professional football, having won 47 team trophies across more than 1,200 career appearances. He has scored over 930 senior career goals and provided over 420 assists

### observation
Successfully implemented LangChain's WikipediaRetriever to retrieve documents based on a user query. For the query "Lionel Messi", the retriever returned three relevant Wikipedia documents covering his biography, rivalry with Cristiano Ronaldo, and professional career. The retrieved content and source URLs demonstrate how external knowledge can be accessed dynamically for a RAG application.


## Task 3: Vector Store Retriever
1. Load documents (Wikipedia or custom text).
2. Create embeddings using OpenAI.
3. Store embeddings in a vector store (FAISS or Chroma).
4. Perform similarity search using a retriever.


In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(wiki_docs,embedding_model)

print("FAISS vector store created successfully.")
print("Number of documents:", vectorstore.index.ntotal)

FAISS vector store created successfully.
Number of documents: 3


In [13]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 2})

print("FAISS retriever created successfully.")

FAISS retriever created successfully.


In [14]:
query = "Which football clubs did Lionel Messi play for?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:700])
    print("-" * 80)


Result 1
Source: https://en.wikipedia.org/wiki/Career_of_Lionel_Messi
Content: Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. His individual achievements include eight Ballon d'Or awards, the most for any footballer. With 47 team trophies, he is the most decorated player in the history of professional football.
Messi's club career began with Barcelona, where he rose through the youth ranks, making his first-team debut in 2004. During his tenure, Barcelona secured ten La Liga titles, seven Copa del Reys, four UEFA Champions Leagues, seven Supercopa de Españas, three UEFA Super Cups, and three FIFA Club World Cups, and Messi became the club's all-time top scorer. In 2021, due to club 
--------------------------------------------------------------------------------

Result 2
Source: https://en.wikipedia.org/wiki/Lionel_Messi
Content: Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine profes

### observation

Successfully generated OpenAI embeddings for three Wikipedia documents and stored them in a FAISS vector store. Created a similarity-based retriever using LangChain and retrieved the two most relevant documents for a user query. The results demonstrate how vector stores enable semantic retrieval of relevant information.


# PART 3 — Advanced Retrieval Strategies



## Task 4: Maximal Marginal Relevance (MMR) Retriever

1. Use search_type="mmr".
2. Compare results with normal similarity search.
3. Explain how MMR improves diversity.

In [15]:
mmr_retriever = vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 2,"fetch_k": 3,"lambda_mult": 0.5})

print("MMR retriever created successfully.")

MMR retriever created successfully.


We chose lambda_mult 0.5 as a balanced starting point. It helps in retriever return relevant documents while avoiding documents that contain almost identical information

In [16]:
query = "Tell me about Lionel Messi's football career and achievements"

# Normal similarity search
similarity_results = retriever.invoke(query)

# MMR search
mmr_results = mmr_retriever.invoke(query)

print("NORMAL SIMILARITY SEARCH")
for i, doc in enumerate(similarity_results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:400])

print("\n\nMMR SEARCH")
for i, doc in enumerate(mmr_results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:400])

NORMAL SIMILARITY SEARCH

Result 1:
Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. His individual achievements include eight Ballon d'Or awards, the most for any footballer. With 47 team trophies, he is the most decorated player in the history of professional football.
Messi's club career began with Barcelona, where he rose through the youth rank

Result 2:
Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. Widely regarded as one of the greatest players in history, he played for the Argentina national team for 21 years. During his career, he has set numerous records for individual accolades won, including eight Ballons d'Or, six Europe


MMR SEARCH

Result 1:
Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter

### observation

- Implemented Maximal Marginal Relevance (MMR) retrieval using LangChain with search_type="mmr".
- Compared its results with normal similarity search. Normal similarity search returned two documents with overlapping information about Messi, whereas MMR returned a document about the Messi–Ronaldo rivalry as the second result.
- This demonstrates how MMR balances relevance and diversity, reducing redundant results and providing broader information.

# Part 3 — Advanced Retrieval Strategies

## Task 5: Multi-Query Retriever
1. Use MultiQueryRetriever.
2. Allow LLM to generate multiple reformulated queries.
3. Retrieve documents for all queries.
4. Compare retrieval quality.


In [17]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),llm=chat_model)

print("Multi-Query Retriever initialized successfully.")

ModuleNotFoundError: No module named 'langchain.retrievers'

In [18]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /Users/bandaanusha/Desktop/GenAI/.venv/bin/python
Python version: 3.11.14 (main, Nov 13 2025, 12:34:27) [Clang 13.1.6 (clang-1316.0.21.2.5)]


In [19]:
%pip install -qU langchain langchain-community langchain-openai


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 2}
    ),
    llm=chat_model
)

print("Multi-Query Retriever initialized successfully.")

ModuleNotFoundError: No module named 'langchain.retrievers'

In [21]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 2}
    ),
    llm=chat_model
)

print("Multi-Query Retriever initialized successfully.")

Multi-Query Retriever initialized successfully.


In [22]:
query = "What are Lionel Messi's major football achievements?"

multi_query_results = multi_query_retriever.invoke(query)

print("Number of documents retrieved:", len(multi_query_results))

for i, doc in enumerate(multi_query_results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:600])
    print("-" * 80)

Number of documents retrieved: 2

Result 1
Source: https://en.wikipedia.org/wiki/Career_of_Lionel_Messi
Content: Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. His individual achievements include eight Ballon d'Or awards, the most for any footballer. With 47 team trophies, he is the most decorated player in the history of professional football.
Messi's club career began with Barcelona, where he rose through the youth ranks, making his first-team debut in 2004. During his tenure, Barcelona secured ten La Liga titles, seven Copa del Reys, four UEFA Champions Leagues, seven Supercopa de Españas, three UEFA Super Cups, an
--------------------------------------------------------------------------------

Result 2
Source: https://en.wikipedia.org/wiki/Lionel_Messi
Content: Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major

In [23]:
query = "What are Lionel Messi's major football achievements?"

# Normal similarity search
similarity_results = retriever.invoke(query)

print("NORMAL SIMILARITY SEARCH")
for i, doc in enumerate(similarity_results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:400])

print("\nMULTI-QUERY RETRIEVER")
for i, doc in enumerate(multi_query_results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content[:400])

NORMAL SIMILARITY SEARCH

Result 1:
Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. His individual achievements include eight Ballon d'Or awards, the most for any footballer. With 47 team trophies, he is the most decorated player in the history of professional football.
Messi's club career began with Barcelona, where he rose through the youth rank

Result 2:
Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. Widely regarded as one of the greatest players in history, he played for the Argentina national team for 21 years. During his career, he has set numerous records for individual accolades won, including eight Ballons d'Or, six Europe

MULTI-QUERY RETRIEVER

Result 1:
Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer 

### observation- 
- Implemented MultiQueryRetriever using LangChain and an OpenAI chat model.
- The retriever generated alternative queries and combined the retrieved documents, removing duplicates. Compared with normal similarity search, both methods returned the same two relevant documents for the Messi achievements query.
- This demonstrates that Multi-Query can broaden retrieval by exploring different query formulations, although improvements depend on the query and available documents.

## Task 6: Contextual Compression Retriever
1. Use ContextualCompressionRetriever.
2. Combine base retriever with a document compressor.
3. Reduce irrelevant content.
4. Show before vs after compression.


In [24]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


In [25]:
compressor = LLMChainExtractor.from_llm(chat_model)


In [26]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

compression_retriever = ContextualCompressionRetriever(base_compressor=compressor,base_retriever=base_retriever)


In [27]:
query = "What awards and individual achievements has Lionel Messi received?"

# Retrieve original documents
original_docs = base_retriever.invoke(query)

# Retrieve compressed documents
compressed_docs = compression_retriever.invoke(query)

print("BEFORE COMPRESSION")

for i, doc in enumerate(original_docs, start=1):
    print(f"\nResult {i}")
    print("Content:", doc.page_content[:800])

print("\n\nAFTER COMPRESSION")

for i, doc in enumerate(compressed_docs, start=1):
    print(f"\nResult {i}")
    print("Content:", doc.page_content)

BEFORE COMPRESSION

Result 1
Content: Lionel Messi is an Argentine professional footballer who plays as a forward for and captains the Major League Soccer club Inter Miami. His individual achievements include eight Ballon d'Or awards, the most for any footballer. With 47 team trophies, he is the most decorated player in the history of professional football.
Messi's club career began with Barcelona, where he rose through the youth ranks, making his first-team debut in 2004. During his tenure, Barcelona secured ten La Liga titles, seven Copa del Reys, four UEFA Champions Leagues, seven Supercopa de Españas, three UEFA Super Cups, and three FIFA Club World Cups, and Messi became the club's all-time top scorer. In 2021, due to club financial constraints and league regulations, Messi left Barcelona and signed with Paris Saint-Germa

Result 2
Content: Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains the Major League

In [28]:
original_length = sum(len(doc.page_content) for doc in original_docs)
compressed_length = sum(len(doc.page_content) for doc in compressed_docs)

print("Original content length:", original_length)
print("Compressed content length:", compressed_length)

if original_length > 0:
    reduction = (1 - compressed_length / original_length) * 100
    print(f"Content reduction: {reduction:.2f}%")

Original content length: 12000
Compressed content length: 3394
Content reduction: 71.72%


### observation

- Implemented ContextualCompressionRetriever using LangChain's LLMChainExtractor and an OpenAI chat model.
- The compressor filtered retrieved documents based on their relevance to the user's query.
- The original content of 12,000 characters was reduced to 3,394 characters, achieving a 71.72% reduction. Relevant information about Messi's awards, achievements and career was retained, while less relevant content was removed. This improves retrieval efficiency by reducing unnecessary context passed to the LLM.


# Part 4 — YouTube Content RAG Chatbot (Mini Project)


## Task 7: Load YouTube Content
1. Use YoutubeLoader to load transcript from a YouTube video.
2. Split transcript using text splitters.


In [29]:
%pip install -qU youtube-transcript-api


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
from langchain_community.document_loaders import YoutubeLoader

youtube_url = "https://youtu.be/dBzMhxa1u94"

youtube_loader = YoutubeLoader.from_youtube_url(youtube_url,add_video_info=False)

youtube_docs = youtube_loader.load()

print("Transcript loaded successfully.")
print("Number of documents:", len(youtube_docs))
print("\nTranscript preview:\n")
print(youtube_docs[0].page_content[:2000])

Transcript loaded successfully.
Number of documents: 1

Transcript preview:

[music] >> Oh, wow. Yes. >> All the signs that I am I have to hold on to the oh handle. >> That means you're getting better at this. >> This is interesting. Just for the record, the Kushaq left my heart. Skoda asked us if we wanted to be part of their upcoming event, GOAT, greatest on a track. The premise was simple. Drive a Skoda on the racetrack and experience what makes it great there. We thought about it, and then we said, "Yes." Provided we get not one, not two, but three cars. I know you're thinking of the VRS badge, cars that [music] trade a little bit of their everyday usability for greater on-track ability, cars like a Skoda Octavia RS or the upcoming Kodiaq cars, >> [music] >> or maybe a top-secret project like the Kushaq cars. But the cars that I had in mind are cars that are greater still because there are greater numbers of them across our country, greater numbers of them in garages that you have.

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=150)

youtube_chunks = text_splitter.split_documents(youtube_docs)

print("Number of chunks:", len(youtube_chunks))
print("\nFirst chunk:\n")
print(youtube_chunks[0].page_content)

Number of chunks: 13

First chunk:

[music] >> Oh, wow. Yes. >> All the signs that I am I have to hold on to the oh handle. >> That means you're getting better at this. >> This is interesting. Just for the record, the Kushaq left my heart. Skoda asked us if we wanted to be part of their upcoming event, GOAT, greatest on a track. The premise was simple. Drive a Skoda on the racetrack and experience what makes it great there. We thought about it, and then we said, "Yes." Provided we get not one, not two, but three cars. I know you're thinking of the VRS badge, cars that [music] trade a little bit of their everyday usability for greater on-track ability, cars like a Skoda Octavia RS or the upcoming Kodiaq cars, >> [music] >> or maybe a top-secret project like the Kushaq cars. But the cars that I had in mind are cars that are greater still because there are greater numbers of them across our country, greater numbers of them in garages that you have. The Kushaq, the Kushaq, the Slavia. SUV,

### observation
- Successfully loaded the YouTube video transcript using LangChain's YoutubeLoader.
- The transcript was retrieved as a single document and split into 13 smaller chunks using RecursiveCharacterTextSplitter with a chunk size of 1,000 characters and an overlap of 150 characters.
- These chunks will be used to create a vector store for semantic search in the next task.

## Task 8: Build Vector Store for YouTube Content
1. Generate embeddings using OpenAI.
2. Store embeddings in FAISS or Chroma.
3. Create a retriever.


In [32]:
youtube_vectorstore = FAISS.from_documents(youtube_chunks,embedding_model)

print("YouTube FAISS vector store created successfully.")
print("Number of chunks indexed:", youtube_vectorstore.index.ntotal)

YouTube FAISS vector store created successfully.
Number of chunks indexed: 13


In [33]:
youtube_retriever = youtube_vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 3})


## Task 9: Build YouTube RAG Chatbot

Create a chatbot that:
1. Accepts user questions.
2. Retrieves relevant transcript chunks.
3. Uses OpenAI LLM to generate answers.
4. Maintains conversation context.


In [34]:
from langchain_core.prompts import ChatPromptTemplate

youtube_rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant that answers questions about a YouTube video.

        Use only the provided video transcript context to answer.
        If the answer is not available in the transcript, clearly say that
        you could not find the information in the video.

        Keep answers clear and concise.

        Previous conversation:
        {chat_history}

        Video transcript context:
        {context}
        """
    ),
    ("human", "{question}")
])

print("YouTube RAG prompt created successfully.")

YouTube RAG prompt created successfully.


In [35]:
def ask_video(question):
    # Retrieve relevant transcript chunks
    retrieved_docs = youtube_retriever.invoke(question)

    # Combine retrieved content
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Send question and context to the LLM
    response = (youtube_rag_prompt | chat_model).invoke({
        "question": question,
        "context": context,
        "chat_history": chat_history
    })

    # Store conversation history
    chat_history.append(f"Human: {question}")
    chat_history.append(f"Assistant: {response.content}")

    return response.content


chat_history = []

print("YouTube RAG chatbot is ready!")

YouTube RAG chatbot is ready!


### observation
- Developed a YouTube RAG chatbot using LangChain, FAISS and GPT-4o-mini.
- The chatbot retrieves relevant transcript chunks and generates context-grounded answers. Conversation history is maintained to support follow-up questions.
- Testing demonstrated accurate responses to video-related questions and appropriate handling of questions whose answers were not available in the transcript.

## Task 10: Testing & Evaluation
1. Ask at least 5 questions related to the video.
2. Verify answers come from video content.
3. Handle unknown questions gracefully.

In [41]:
question = "What is the main purpose of Skoda's GOAT event?"

answer = ask_video(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: What is the main purpose of Skoda's GOAT event?

Answer: The main purpose of Skoda's GOAT event is to drive a Skoda on the racetrack and experience what makes the cars great in that environment.


In [42]:
question = "What are the three Skoda cars featured in the event, and what engine do they share?"

answer = ask_video(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: What are the three Skoda cars featured in the event, and what engine do they share?

Answer: The three Skoda cars featured in the event are the Kushaq, the Kushaq, and the Slavia. They all share the same 1-liter engine, producing 115 PS and 170 Newton meters of torque.


In [43]:
question = "Which of these cars is the sedan?"

answer = ask_video(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: Which of these cars is the sedan?

Answer: The sedan featured in the event is the Slavia.


In [44]:
question = "What is the on-road price of the Skoda Slavia in Hyderabad?"

answer = ask_video(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: What is the on-road price of the Skoda Slavia in Hyderabad?

Answer: I could not find the information in the video.


In [40]:
question = "Why is the Kushaq considered suitable for the racetrack?"

answer = ask_video(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: Why is the Kushaq considered suitable for the racetrack?

Answer: The Kushaq is considered suitable for the racetrack because it is the smallest of the three Skoda cars featured, with a compact length and the smallest wheelbase, which contributes to its agility. This agility is great for the racetrack, allowing it to hug curves effectively and carry more speed through corners. Additionally, it shares the same 1-liter engine as the other cars but has an 8-speed automatic gearbox, which helps it utilize its power better.


### observation
- Evaluated the YouTube RAG chatbot using five questions covering factual retrieval, multiple details, conversational follow-ups, and out-of-context queries.
- The chatbot successfully retrieved relevant information from the transcript, maintained conversation context, and avoided fabricating answers when information was unavailable.
- Minor limitations were observed due to transcription inaccuracies and occasional unsupported details in generated responses. Overall, the RAG pipeline demonstrated effective retrieval and question-answering capabilities.